In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [2]:
!pip install -q -U transformers accelerate bitsandbytes datasets pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.8 MB/s eta 0:00:00


In [2]:
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

transformers: 5.16.1
torch: 2.11.0+cu128


In [3]:
import os
USE_DRIVE = False   # True = survive disconnects (recommended for the long full run)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/hallusionbench_results"
else:
    RESULTS_DIR = "/content/results"

os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_PATH = f"{RESULTS_DIR}/internvl3_8b_results.json"
print("Saving to:", RESULTS_PATH)

Saving to: /content/results/internvl3_8b_results.json


In [4]:
from datasets import load_dataset
ds = load_dataset("lmms-lab/HallusionBench")
print(ds)

DatasetDict({
    image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 951
    })
    non_image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 178
    })
})


In [5]:
import torch, gc
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "OpenGVLab/InternVL3-8B-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
).eval()

print("Loaded.")
print(processor.image_processor)

model.safetensors.index.json:   0%|          | 0.00/73.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/781 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

Loaded.
GotOcr2ImageProcessor {
  "crop_to_patches": false,
  "data_format": "channels_first",
  "default_to_square": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.485,
    0.456,
    0.406
  ],
  "image_processor_type": "GotOcr2ImageProcessor",
  "image_std": [
    0.229,
    0.224,
    0.225
  ],
  "max_patches": 12,
  "min_patches": 1,
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 448,
    "width": 448
  }
}



In [6]:
import importlib.metadata as md
try:
    print("bitsandbytes:", md.version("bitsandbytes"))
except md.PackageNotFoundError:
    print("bitsandbytes: NOT INSTALLED")

bitsandbytes: NOT INSTALLED


In [7]:
!pip install -q -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.7 MB/s eta 0:00:00


In [6]:
if hasattr(processor.image_processor, "max_patches"):
    processor.image_processor.max_patches = 6
    print("max_patches set to", processor.image_processor.max_patches)
else:
    print("No max_patches attribute, paste the image_processor printout")

max_patches set to 6


In [7]:
def parse_yes_no(text):
    a = text.strip().lower()
    if a.startswith("yes"): return "1"
    if a.startswith("no"): return "0"
    return "unclear"

def build_inputs(question, image=None):
    content = []
    if image is not None:
        content.append({"type": "image"})
    content.append({"type": "text", "text": f"{question} Please answer with only Yes or No."})
    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if image is not None:
        inputs = processor(text=text, images=image.convert("RGB"), return_tensors="pt")
    else:
        inputs = processor(text=text, return_tensors="pt")

    inputs = inputs.to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
    return inputs

@torch.inference_mode()
def ask(question, image=None):
    inputs = build_inputs(question, image)
    out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    resp = processor.batch_decode(gen, skip_special_tokens=True)[0]
    del inputs, out, gen
    gc.collect()
    torch.cuda.empty_cache()
    return resp

In [8]:
ex = ds["image"][0]
r = ask(ex["question"], ex["image"])
print("IMAGE Q:", ex["question"][:80], "\n  GT:", ex["gt_answer"], "| Response:", repr(r), "| Parsed:", parse_yes_no(r))

ex2 = ds["non_image"][0]
r2 = ask(ex2["question"], None)
print("TEXT Q:", ex2["question"][:80], "\n  GT:", ex2["gt_answer"], "| Response:", repr(r2), "| Parsed:", parse_yes_no(r2))

IMAGE Q: Is China, Hongkong SAR, the leading importing country of gold, silverware, and j 
  GT: 0 | Response: 'Yes.' | Parsed: 1
TEXT Q: Is China, Hongkong SAR, the leading importing country of gold, silverware, and j 
  GT: 0 | Response: 'Yes.' | Parsed: 1


In [9]:
import time
start = time.time()
for ex in ds["image"].select(range(20)):
    ask(ex["question"], ex["image"])
el = time.time() - start
print(f"20 questions: {el:.1f}s | estimated full 1129: {el*1129/20/60:.1f} min")

20 questions: 51.0s | estimated full 1129: 48.0 min


In [10]:
import json

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
else:
    results = []

done = {(r["question_id"], r["split"]) for r in results}

def save():
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)

def run_split(split_name):
    n = len(ds[split_name])
    for i, ex in enumerate(ds[split_name]):
        if (ex["question_id"], split_name) in done:
            continue
        try:
            resp = ask(ex["question"], ex["image"])
        except Exception as e:
            gc.collect(); torch.cuda.empty_cache()
            resp = f"ERROR: {type(e).__name__}: {str(e)[:100]}"

        results.append({
            "question_id": ex["question_id"],
            "split": split_name,
            "category": ex["category"],
            "subcategory": ex["subcategory"],
            "figure_id": ex["figure_id"],
            "set_id": ex["set_id"],
            "question": ex["question"],
            "gt_answer": ex["gt_answer"],
            "raw_response": resp,
            "model_prediction": parse_yes_no(resp),
        })
        if (i + 1) % 50 == 0:
            save()
            print(f"[{split_name}] {i+1}/{n}")
    save()

run_split("image")
run_split("non_image")
print("Done:", len(results))

[image] 50/951
[image] 100/951
[image] 150/951
[image] 200/951
[image] 250/951
[image] 300/951
[image] 350/951
[image] 400/951
[image] 450/951
[image] 500/951


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[image] 550/951
[image] 600/951
[image] 650/951
[image] 700/951
[image] 750/951
[image] 800/951
[image] 850/951
[image] 900/951
[image] 950/951
[non_image] 50/178
[non_image] 100/178
[non_image] 150/178
Done: 1129


In [11]:
from collections import defaultdict

def compute_metrics(rs):
    total = len(rs)
    if total == 0: return None
    ok = lambda r: r["model_prediction"] == str(r["gt_answer"])

    aAcc = sum(ok(r) for r in rs) / total * 100

    sets = defaultdict(list)
    for r in rs: sets[(r["category"], r["subcategory"], r["set_id"])].append(r)
    qAcc = sum(all(ok(m) for m in v) for v in sets.values()) / len(sets) * 100

    figs = defaultdict(list)
    for r in rs: figs[(r["category"], r["subcategory"], r["figure_id"])].append(r)
    fAcc = sum(all(ok(m) for m in v) for v in figs.values()) / len(figs) * 100

    return {"aAcc": round(aAcc, 2), "qAcc": round(qAcc, 2), "fAcc": round(fAcc, 2),
            "n_questions": total, "n_sets": len(sets), "n_figures": len(figs)}

print("=== Overall ===")
print(compute_metrics(results))

by_cat = defaultdict(list)
for r in results: by_cat[r["category"]].append(r)
print("\n=== By Category ===")
for k, v in by_cat.items(): print(k, compute_metrics(v))

by_sub = defaultdict(list)
for r in results: by_sub[(r["category"], r["subcategory"])].append(r)
print("\n=== By Category + Subcategory ===")
for k, v in sorted(by_sub.items()): print(k, compute_metrics(v))

print("\nUnclear/error responses:", sum(r["model_prediction"] == "unclear" for r in results), "/", len(results))

=== Overall ===
{'aAcc': 63.95, 'qAcc': 11.38, 'fAcc': 3.85, 'n_questions': 1129, 'n_sets': 167, 'n_figures': 26}

=== By Category ===
VS {'aAcc': 71.0, 'qAcc': 3.51, 'fAcc': 0.0, 'n_questions': 538, 'n_sets': 57, 'n_figures': 13}
VD {'aAcc': 57.53, 'qAcc': 15.45, 'fAcc': 7.69, 'n_questions': 591, 'n_sets': 110, 'n_figures': 13}

=== By Category + Subcategory ===
('VD', 'figure') {'aAcc': 68.75, 'qAcc': 20.0, 'fAcc': 33.33, 'n_questions': 80, 'n_sets': 20, 'n_figures': 3}
('VD', 'illusion') {'aAcc': 63.89, 'qAcc': 12.9, 'fAcc': 0.0, 'n_questions': 144, 'n_sets': 31, 'n_figures': 2}
('VD', 'math') {'aAcc': 42.59, 'qAcc': 0.0, 'fAcc': 0.0, 'n_questions': 108, 'n_sets': 18, 'n_figures': 2}
('VD', 'ocr') {'aAcc': 77.53, 'qAcc': 42.86, 'fAcc': 0.0, 'n_questions': 89, 'n_sets': 21, 'n_figures': 3}
('VD', 'video') {'aAcc': 45.88, 'qAcc': 0.0, 'fAcc': 0.0, 'n_questions': 170, 'n_sets': 20, 'n_figures': 3}
('VS', 'chart') {'aAcc': 71.36, 'qAcc': 9.09, 'fAcc': 0.0, 'n_questions': 206, 'n_sets': 

In [12]:
metrics_summary = {
    "model": "internvl3-8b-hf",
    "overall": compute_metrics(results),
    "by_category": {c: compute_metrics(v) for c, v in by_cat.items()},
    "by_category_subcategory": {f"{k[0]}_{k[1]}": compute_metrics(v) for k, v in by_sub.items()},
}
METRICS_PATH = f"{RESULTS_DIR}/internvl3_8b_metrics.json"
with open(METRICS_PATH, "w") as f:
    json.dump(metrics_summary, f, indent=2)

from google.colab import files
files.download(RESULTS_PATH)
files.download(METRICS_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>